# Notebook 3: Combined Drug Interaction Query Tool

Loads all trained models from Google Drive and provides an interactive widget UI for:

- **Drug–Drug Interaction mode**: Input two drugs (by name or SMILES) → compare:
  1. **Random Forest prediction** (89-class, Morgan fingerprints)
  2. **GNN prediction** (top-5 classes, molecular graph encoding)
  3. **Food behavior** for each drug (lookup table or model)

- **Single Drug mode**: Input one drug → food behavior only

- **Model Comparison section**: Head-to-head RF vs GNN accuracy on held-out samples

### Prerequisites
Run **Notebook 1** and **Notebook 2** first to save models to:
`/content/drive/MyDrive/drug_interaction_models/`


In [ ]:
# ─── Install Dependencies ───────────────────────────────────────────────────
!pip install -q --upgrade pip
!pip uninstall -y numpy # Force uninstall any existing numpy
!pip install -q numpy==1.26.0
!pip install -q rdkit transformers torch ipywidgets

Found existing installation: numpy 2.4.4
Uninstalling numpy-2.4.4:
  Successfully uninstalled numpy-2.4.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.0 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.0 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.0 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.0 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.0 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.0 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.0 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2;

In [ ]:
# ─── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pickle, json, os, time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

from transformers import AutoTokenizer, AutoModel
from rdkit import Chem
from rdkit.Chem import AllChem

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')


In [ ]:
# ─── Mount Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

MODEL_DIR = '/content/drive/MyDrive/drug_interaction_models'
print('Model directory:', MODEL_DIR)
print('Files:', os.listdir(MODEL_DIR))

---
## Module A — DDI Random Forest Model
Predicts: *Which of the 89 interaction mechanism types applies?*


In [ ]:
!pip install scikit-learn==1.2.2

In [ ]:
# ─── Module A — DDI Multi-class Model ───────────────────────────────────────
import pickle, json, os
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem

with open(os.path.join(MODEL_DIR, "ddi_multiclass_model.pkl"), "rb") as f:
    ddi_model = pickle.load(f)
with open(os.path.join(MODEL_DIR, "ddi_label_encoder.pkl"), "rb") as f:
    ddi_le = pickle.load(f)
with open(os.path.join(MODEL_DIR, "ddi_label_map.json")) as f:
    DDI_LABEL_MAP_raw = json.load(f)
    DDI_LABEL_MAP = {int(k): v for k, v in DDI_LABEL_MAP_raw.items()}

print("=== DDI Multi-class model ===")
print(f"  Classes: {len(ddi_le.classes_)} interaction types")

def smiles_to_fp(smiles, radius=2, nbits=2048):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None: return None
    return np.array(AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits))

def predict_ddi_type(smiles1, smiles2, top_k=3):
    """
    MODULE A: Predict the interaction mechanism type(s) for two drugs.
    Returns: list of (interaction_description, probability) sorted by prob desc.
    An empty list means the pair may not interact (no mechanism predicted confidently).
    """
    fp1, fp2 = smiles_to_fp(smiles1), smiles_to_fp(smiles2)
    if fp1 is None or fp2 is None:
        return []
    X = np.hstack([fp1, fp2]).reshape(1, -1)
    probs = ddi_model.predict_proba(X)[0]          # shape: (n_classes,)
    top_enc = np.argsort(probs)[::-1][:top_k]      # top-k encoded class indices
    results = []
    for enc_idx in top_enc:
        orig_label = int(ddi_le.classes_[enc_idx])  # original Y value (1-indexed)
        description = DDI_LABEL_MAP.get(orig_label, f"Type {orig_label}")
        results.append((description, float(probs[enc_idx])))
    return results

# Smoke test
test_r = predict_ddi_type("CC(=O)Oc1ccccc1C(=O)O", "OC(=O)Cc1ccc(cc1)n1ccnc1", top_k=3)
print("Smoke test top predictions:")
for desc, p in test_r:
    print(f"  {p:.3f}  {desc[:80]}")

---
## Module B — DDI GNN Model
Predicts: *Which of the top-5 interaction types applies?* (GCN + Morgan FP, same DDI dataset)


In [ ]:
!pip install torch-geometric

In [ ]:
# ─── Module B — DDI GNN Model ────────────────────────────────────────────────
# Rebuild GNN architecture (must match Notebook 1 exactly)

with open(os.path.join(MODEL_DIR, 'ddi_gnn_config.json')) as f:
    gnn_cfg = json.load(f)
with open(os.path.join(MODEL_DIR, 'ddi_gnn_label_encoder.pkl'), 'rb') as f:
    gnn_le = pickle.load(f)
with open(os.path.join(MODEL_DIR, 'ddi_gnn_label_map.json')) as f:
    GNN_LABEL_MAP = {int(k): v for k, v in json.load(f).items()}

GNN_N_CLASSES    = gnn_cfg['n_classes']
GNN_HIDDEN       = gnn_cfg['hidden']
GNN_EMBED        = gnn_cfg['embed']
GNN_NODE_FEAT_DIM = gnn_cfg['node_feat_dim']
GNN_FP_DIM       = gnn_cfg['fp_dim']
GNN_ATOM_TYPES   = gnn_cfg['atom_types']


class MolGNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(GNN_NODE_FEAT_DIM, GNN_HIDDEN)
        self.conv2 = GCNConv(GNN_HIDDEN, GNN_HIDDEN)
        self.conv3 = GCNConv(GNN_HIDDEN, GNN_HIDDEN)
        self.fc = nn.Linear(GNN_HIDDEN + GNN_FP_DIM, GNN_EMBED)

    def forward(self, data):
        x, ei, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.conv1(x, ei))
        x = F.relu(self.conv2(x, ei))
        x = F.relu(self.conv3(x, ei))
        x = global_mean_pool(x, batch)
        fp = data.fp.view(x.size(0), -1)
        return F.relu(self.fc(torch.cat([x, fp], dim=1)))


class DDIGNNModel(nn.Module):
    def __init__(self, n_classes=GNN_N_CLASSES):
        super().__init__()
        self.encoder = MolGNN()
        self.classifier = nn.Sequential(
            nn.Linear(GNN_EMBED * 2, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, n_classes)
        )

    def forward(self, mol_a, mol_b):
        return self.classifier(torch.cat([self.encoder(mol_a), self.encoder(mol_b)], dim=1))


ddi_gnn = DDIGNNModel().to(DEVICE)
ddi_gnn.load_state_dict(torch.load(
    os.path.join(MODEL_DIR, 'ddi_gnn_weights.pt'), map_location=DEVICE
))
ddi_gnn.eval()
print(f'=== DDI GNN Model ===')
print(f'  Classes: {GNN_N_CLASSES} (top-5 interaction types)')
print(f'  CV mean accuracy: {gnn_cfg.get("mean_cv_acc", "N/A")}')


def smiles_to_gnn_graph(smiles):
    """Convert SMILES to PyG Data object (must match Notebook 1)."""
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    node_feats = []
    for atom in mol.GetAtoms():
        one_hot = [int(atom.GetSymbol() == t) for t in GNN_ATOM_TYPES]
        one_hot.append(int(atom.GetSymbol() not in GNN_ATOM_TYPES))
        one_hot += [atom.GetDegree() / 6.0, int(atom.GetIsAromatic())]
        node_feats.append(one_hot)
    x = torch.tensor(node_feats, dtype=torch.float)
    edge_index = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index += [[i, j], [j, i]]
    if not edge_index:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    gen = AllChem.GetMorganGenerator(radius=2, fpSize=GNN_FP_DIM)
    fp = torch.tensor(list(gen.GetFingerprint(mol)), dtype=torch.float)
    return Data(x=x, edge_index=edge_index, fp=fp)


def predict_ddi_gnn(smiles1, smiles2, top_k=3):
    """
    MODULE B: Predict interaction type using the GNN.
    Returns list of (interaction_description, probability) from the top-5 class model.
    """
    g1, g2 = smiles_to_gnn_graph(smiles1), smiles_to_gnn_graph(smiles2)
    if g1 is None or g2 is None:
        return []
    with torch.no_grad():
        b1 = Batch.from_data_list([g1]).to(DEVICE)
        b2 = Batch.from_data_list([g2]).to(DEVICE)
        probs = torch.softmax(ddi_gnn(b1, b2), dim=1).cpu().numpy()[0]
    top_idx = np.argsort(probs)[::-1][:top_k]
    return [(GNN_LABEL_MAP.get(int(i), f'Class {i}'), float(probs[i])) for i in top_idx]


# Smoke test
_r = predict_ddi_gnn('CC(=O)Oc1ccccc1C(=O)O', 'OC(=O)Cc1ccc(cc1)n1ccnc1')
print('GNN smoke test:', [(d[:50], f'{p:.3f}') for d, p in _r])
print('DDI GNN module ready.')


---
## Module C — Food Behavior Model + Lookup Table
Predicts: *What food behaviors apply to this drug?*

In [ ]:
# ─── Load Food Model & Lookup ────────────────────────────────────────────────
with open(os.path.join(MODEL_DIR, 'food_behavior_model.pkl'), 'rb') as f:
    food_model = pickle.load(f)

with open(os.path.join(MODEL_DIR, 'food_mlb.pkl'), 'rb') as f:
    food_mlb = pickle.load(f)

with open(os.path.join(MODEL_DIR, 'food_lookup.json')) as f:
    FOOD_LOOKUP = json.load(f)  # {drug_name: {smiles, food_interactions, behavior_labels}}

with open(os.path.join(MODEL_DIR, 'food_categories.json')) as f:
    FOOD_CATEGORIES = json.load(f)

# Load BERT model for food text encoding (needed when using Model prediction mode)
with open(os.path.join(MODEL_DIR, 'bert_model_name.txt')) as f:
    BERT_MODEL_NAME = f.read().strip()

print('✅ Food model and lookup loaded')
print(f'  Lookup table size: {len(FOOD_LOOKUP)} drugs')
print(f'  Food categories: {FOOD_CATEGORIES}')
print(f'  BERT model: {BERT_MODEL_NAME}')

In [ ]:
# ─── Load BERT for food prediction mode ──────────────────────────────────────
print(f'Loading BioBERT ({BERT_MODEL_NAME})...')
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
bert_model_obj = AutoModel.from_pretrained(BERT_MODEL_NAME).to(DEVICE)
bert_model_obj.eval()
print('BioBERT ready.')

def bert_encode_single(text, max_length=128):
    enc = bert_tokenizer(
        [text], padding=True, truncation=True,
        max_length=max_length, return_tensors='pt'
    ).to(DEVICE)
    with torch.no_grad():
        out = bert_model_obj(**enc)
    mask = enc['attention_mask'].unsqueeze(-1).float()
    emb  = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
    return emb.cpu().numpy()

# Build reverse lookup: smiles → drug name
SMILES_TO_NAME = {}
for drug_name, info in FOOD_LOOKUP.items():
    if info.get('smiles'):
        SMILES_TO_NAME[info['smiles']] = drug_name

def smiles_to_fp(smiles, radius=2, nbits=2048):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None: return None
    return np.array(AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits))

def predict_food_model(smiles):
    fp = smiles_to_fp(smiles)
    if fp is None:
        return []
    bert_emb = np.zeros((1, 768))
    X = np.hstack([fp.reshape(1, -1), bert_emb])
    y_pred = food_model.predict(X)[0]
    labels = food_mlb.inverse_transform(y_pred.reshape(1, -1))[0]
    return list(labels)

def lookup_food_by_smiles(smiles):
    if smiles in SMILES_TO_NAME:
        name = SMILES_TO_NAME[smiles]
        return FOOD_LOOKUP[name]['food_interactions'], name
    return None, None

def lookup_food_by_name(drug_name):
    drug_name_l = drug_name.lower()
    for name, info in FOOD_LOOKUP.items():
        if drug_name_l in name.lower() or name.lower() in drug_name_l:
            return info['food_interactions'], name
    return None, None

print('Food module ready.')


In [ ]:
# ─── PubChemPy: Convert Drug Name → SMILES ───────────────────────────────────
!pip install -q pubchempy
import pubchempy as pcp

_pubchem_cache = {}

def name_to_smiles(name):
    """
    Convert a drug name to SMILES via PubChem.
    Returns SMILES string or None. Results cached in-session.
    """
    if not name:
        return None
    key = name.strip().lower()
    if key in _pubchem_cache:
        return _pubchem_cache[key]
    # Also try lookup table first (faster, no API call)
    interactions, found = lookup_food_by_name(name)
    if found and FOOD_LOOKUP[found].get('smiles'):
        smi = FOOD_LOOKUP[found]['smiles']
        _pubchem_cache[key] = smi
        return smi
    try:
        compounds = pcp.get_compounds(name, 'name', timeout=10)
        if compounds:
            smi = compounds[0].isomeric_smiles or compounds[0].canonical_smiles
            _pubchem_cache[key] = smi
            return smi
    except Exception as e:
        print(f'PubChem lookup failed for "{name}": {e}')
    _pubchem_cache[key] = None
    return None

# Test
_smi = name_to_smiles('Aspirin')
print(f'Aspirin SMILES: {_smi}')
print('PubChemPy helper ready.')


---
## Interactive Query UI

Enter a drug by **name** OR **SMILES string** — the tool will auto-convert names to SMILES via PubChem when needed.  
For food lookup mode, a name search directly checks the Kaggle lookup table.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  INTERACTIVE DRUG INTERACTION QUERY UI — RF + GNN SIDE BY SIDE
# ═══════════════════════════════════════════════════════════════════════════

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

style      = {'description_width': '160px'}
layout_wide = widgets.Layout(width='100%')

# ── Mode toggles ──
mode_toggle = widgets.RadioButtons(
    options=['Drug–Drug Interaction', 'Single Drug (Food Only)'],
    value='Drug–Drug Interaction', description='', style=style
)
food_toggle = widgets.RadioButtons(
    options=['📖 Lookup Table (reliable, text-based)',
             '🔮 Model Prediction (from SMILES structure)'],
    value='📖 Lookup Table (reliable, text-based)', description='', style=style
)

# ── Drug input boxes — now accept EITHER name OR SMILES ──
drug1_box = widgets.Text(
    placeholder='Drug name (e.g. Aspirin) OR SMILES string',
    description='Drug 1:', style=style, layout=layout_wide
)
drug2_box = widgets.Text(
    placeholder='Drug name (e.g. Warfarin) OR SMILES string',
    description='Drug 2:', style=style, layout=layout_wide
)
hint_label = widgets.HTML(
    '<i style="color:#aaa;font-size:0.85em">💡 You can enter a drug name or a SMILES string. '
    'For food lookup, a name is looked up directly; for model prediction the name is '
    'converted to SMILES via PubChem.</i>'
)

top_k_slider = widgets.IntSlider(
    value=3, min=1, max=10, step=1,
    description='Top-K results:', style=style
)

run_btn = widgets.Button(
    description='▶ Run Query',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='40px')
)
output_area = widgets.Output()

# ── Layout ──
section_layout = widgets.Layout(padding='12px 16px', margin='10px 0px',
                                border='1px solid #333', border_radius='8px')

mode_section = widgets.VBox([
    widgets.HTML('<b style="color:#4fc3f7">Query Mode</b>'), mode_toggle
], layout=section_layout)
food_section = widgets.VBox([
    widgets.HTML('<b style="color:#4fc3f7">Food Behavior Source</b>'), food_toggle
], layout=section_layout)
top_controls = widgets.HBox([mode_section, food_section],
                             layout=widgets.Layout(justify_content='space-between'))

drug2_section = widgets.VBox([drug2_box])

def update_layout(change):
    drug2_section.layout.display = '' if mode_toggle.value == 'Drug–Drug Interaction' else 'none'
mode_toggle.observe(update_layout, names='value')
update_layout(None)

# ── HTML helpers ──
def section_card(title, content_lines, bg='#1a1a2e', title_color='#fff'):
    items = ''.join(f'<li style="margin:3px 0">{l}</li>' for l in content_lines)
    return f"""
    <div style="background:{bg};border-radius:8px;padding:12px 18px;margin:6px 0;color:#eee">
      <strong style="font-size:1.05em;color:{title_color}">{title}</strong>
      <ul style="margin:6px 0 0 0;padding-left:20px">{items}</ul>
    </div>"""

def side_by_side(rf_html, gnn_html):
    return f"""
    <div style="display:flex;gap:12px;margin:6px 0">
      <div style="flex:1">{rf_html}</div>
      <div style="flex:1">{gnn_html}</div>
    </div>"""

def resolve_drug(text):
    """
    Given a name or SMILES string, return (smiles, display_name).
    If it parses as a valid SMILES → use as-is.
    Otherwise treat as a drug name → PubChem lookup.
    """
    text = text.strip()
    if not text:
        return None, ''
    mol = Chem.MolFromSmiles(text)
    if mol is not None:
        # It's a valid SMILES string
        return text, text[:60]
    # Treat as drug name → convert to SMILES
    print(f'Looking up "{text}" on PubChem...')
    smi = name_to_smiles(text)
    return smi, text

def get_food_info(smiles, name, use_lookup):
    if use_lookup:
        interactions, found = (None, None)
        if name and Chem.MolFromSmiles(name.strip()) is None:  # it's a name, not SMILES
            interactions, found = lookup_food_by_name(name)
        if not interactions and smiles:
            interactions, found = lookup_food_by_smiles(smiles)
        if interactions:
            lines = [f'<em>Matched: <b>{found}</b></em>'] + interactions[:6]
        else:
            lines = ['⚠️ Not found in lookup table. Try Model Prediction or check drug name.']
        return lines, 'lookup'
    else:
        if not smiles:
            return ['⚠️ No SMILES available for model prediction.'], 'model'
        cats = predict_food_model(smiles)
        lines = ([f'🔮 <b>{c.replace("_"," ")}</b>' for c in cats]
                 if cats else ['No food categories predicted above threshold.'])
        lines.append('<em style="font-size:0.85em">⚠️ Small dataset — lookup is more reliable</em>')
        return lines, 'model'

# ── Main query handler ──
def on_run(btn):
    with output_area:
        clear_output()
        raw1 = drug1_box.value.strip()
        raw2 = drug2_box.value.strip()
        mode = mode_toggle.value
        topk = top_k_slider.value
        use_lookup = '📖 Lookup' in food_toggle.value

        if not raw1:
            display(HTML('<p style="color:red">⚠️ Please enter Drug 1.</p>'))
            return

        smi1, name1 = resolve_drug(raw1)
        if smi1 is None:
            display(HTML(f'<p style="color:red">❌ Could not find SMILES for "{raw1}". Check the name or enter a SMILES string.</p>'))
            return

        parts = [f'<h3 style="color:#4fc3f7">Query Results</h3>']
        parts.append(f'<p><b>Drug 1:</b> {name1} &nbsp;<code style="font-size:0.8em">{smi1[:60]}</code></p>')

        if mode == 'Drug–Drug Interaction':
            if not raw2:
                display(HTML('<p style="color:red">⚠️ Please enter Drug 2.</p>'))
                return
            smi2, name2 = resolve_drug(raw2)
            if smi2 is None:
                display(HTML(f'<p style="color:red">❌ Could not find SMILES for "{raw2}".</p>'))
                return
            parts.append(f'<p><b>Drug 2:</b> {name2} &nbsp;<code style="font-size:0.8em">{smi2[:60]}</code></p>')

            # ── RF prediction ──
            rf_results = predict_ddi_type(smi1, smi2, top_k=topk)
            if rf_results:
                rf_lines = ['✅ <b>INTERACTION DETECTED</b>'] + \
                           [f'{desc[:75]} <em>({prob:.1%})</em>' for desc, prob in rf_results]
                rf_bg = '#1b3a1b'
            else:
                rf_lines = ['⚪ No mechanism predicted above threshold.']
                rf_bg = '#2a2a2a'
            rf_card = section_card(f'🌲 Random Forest (Top {topk})', rf_lines, rf_bg, '#90ee90')

            # ── GNN prediction ──
            gnn_results = predict_ddi_gnn(smi1, smi2, top_k=topk)
            if gnn_results:
                gnn_lines = ['✅ <b>INTERACTION PREDICTED</b>'] + \
                            [f'{desc[:75]} <em>({prob:.1%})</em>' for desc, prob in gnn_results]
                gnn_bg = '#1a2a3a'
            else:
                gnn_lines = ['⚪ No interaction predicted.']
                gnn_bg = '#2a2a2a'
            gnn_card = section_card(f'🧠 GNN (Top {topk}, top-5 classes)', gnn_lines, gnn_bg, '#87ceeb')

            parts.append('<h4 style="color:#ccc">DDI Predictions — Side by Side</h4>')
            parts.append(side_by_side(rf_card, gnn_card))

            # ── Food behavior ──
            fl1, src = get_food_info(smi1, raw1, use_lookup)
            fl2, _   = get_food_info(smi2, raw2, use_lookup)
            src_label = 'Lookup Table' if src == 'lookup' else 'Model'
            parts.append(section_card(f'🍎 Food Behavior — Drug 1 [{src_label}]', fl1, '#3e2a1a'))
            parts.append(section_card(f'🍎 Food Behavior — Drug 2 [{src_label}]', fl2, '#3e2a1a'))

        else:
            fl, src = get_food_info(smi1, raw1, use_lookup)
            src_label = 'Lookup Table' if src == 'lookup' else 'Model'
            parts.append(section_card(f'🍎 Food Behavior [{src_label}]', fl, '#3e2a1a'))

        display(HTML(''.join(parts)))


run_btn.on_click(on_run)

ui = widgets.VBox([
    widgets.HTML('<h2 style="color:#4fc3f7">🧬 Drug Interaction Query Tool</h2>'),
    widgets.HTML('<hr>'),
    top_controls,
    widgets.HTML('<hr><b>Drug 1</b>'),
    drug1_box,
    widgets.HTML('<b>Drug 2</b> (Drug–Drug mode only)'),
    drug2_section,
    hint_label,
    widgets.HTML('<hr>'),
    widgets.VBox([top_k_slider, run_btn], layout=section_layout),
    widgets.HTML('<hr>'),
    output_area
])
display(ui)


---
## Model Performance Summary


In [ ]:
# ─── Load and Display Saved Performance Plots ────────────────────────────────
from IPython.display import Image

plot_files = {
    'DDI Random Forest — Metrics':         'ddi_metrics_bar.png',
    'DDI Random Forest — Per-class F1':    'ddi_per_class_f1.png',
    'DDI GNN — Training Curves':           'ddi_gnn_training_curves.png',
    'Food Model — Metrics':                'food_metrics_bar.png',
}

for title, fname in plot_files.items():
    fpath = os.path.join(MODEL_DIR, fname)
    if os.path.exists(fpath):
        print(f'\n{"="*60}\n  {title}\n{"="*60}')
        display(Image(fpath, width=700))
    else:
        print(f'[{fname} not found — run Notebook 1 first]')


---
## RF vs GNN Model Comparison

This section runs both models on a shared held-out sample of DDI pairs (drawn from the
top-5 interaction classes that both models were trained/evaluated on) and reports
head-to-head accuracy, per-class F1, confidence calibration, and a verdict on which
method is better suited to this dataset.


In [ ]:
# ─── RF vs GNN Head-to-Head Comparison ───────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from sklearn.calibration import calibration_curve
import seaborn as sns

# ── 1. Build a shared evaluation sample from the DDI top-5 classes ──
# Re-load the DDI dataset (already in memory as train_ddi / val_ddi / test_ddi)
from tdc.multi_pred import DDI

print('Loading DDI data for comparison...')
ddi_data  = DDI(name='DrugBank')
split     = ddi_data.get_split()
full_ddi  = pd.concat([split['train'], split['valid'], split['test']], ignore_index=True)

# Keep only the GNN top-5 classes (so both models can predict on the same label space)
top5_classes = full_ddi['Y'].value_counts().head(5).index
cmp_df = full_ddi[full_ddi['Y'].isin(top5_classes)].copy()

# Balanced sample: 200 per class = 1,000 pairs
cmp_df = cmp_df.groupby('Y', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 200), random_state=99)
).reset_index(drop=True)

print(f'Comparison sample: {len(cmp_df)} pairs, {cmp_df["Y"].nunique()} classes')
print(cmp_df['Y'].value_counts())


In [ ]:
# ─── RF Predictions on Comparison Sample ─────────────────────────────────────
print('Computing RF predictions...')

# Build fingerprint features for the comparison set
def smiles_to_fp_rf(smiles, radius=2, nbits=2048):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None: return np.zeros(nbits)
    return np.array(AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits))

fp1_arr = np.stack(cmp_df['Drug1'].apply(smiles_to_fp_rf).values)
fp2_arr = np.stack(cmp_df['Drug2'].apply(smiles_to_fp_rf).values)
X_cmp   = np.hstack([fp1_arr, fp2_arr])

# RF predicts over 89 classes; we remap to top-5 encoded labels for fair comparison
rf_raw_preds = ddi_model.predict(X_cmp)       # encoded class indices (0-88)
rf_raw_probs = ddi_model.predict_proba(X_cmp)  # shape: (n, 89)

# Map RF predictions to GNN label space (top-5 only)
# gnn_le.classes_ holds the original Y values for the top-5
top5_set    = set(gnn_le.classes_)
# true labels in GNN encoding
y_true_gnn  = []
rf_pred_gnn = []
for i, row in cmp_df.iterrows():
    y_orig = row['Y']
    if y_orig not in top5_set:
        continue
    y_true_gnn.append(int(gnn_le.transform([y_orig])[0]))
    # RF predicted class (original Y value)
    rf_pred_orig = ddi_le.classes_[rf_raw_preds[cmp_df.index.get_loc(i)]]
    if rf_pred_orig in top5_set:
        rf_pred_gnn.append(int(gnn_le.transform([rf_pred_orig])[0]))
    else:
        # RF predicted outside top-5 → assign most likely top-5 via class probs
        top5_rf_idx = [np.where(ddi_le.classes_ == c)[0][0] for c in gnn_le.classes_
                       if c in set(ddi_le.classes_)]
        best = top5_rf_idx[np.argmax(rf_raw_probs[cmp_df.index.get_loc(i)][top5_rf_idx])]
        rf_pred_gnn.append(int(gnn_le.transform([ddi_le.classes_[best]])[0]))

y_true_gnn  = np.array(y_true_gnn)
rf_pred_gnn = np.array(rf_pred_gnn)
print(f'RF prediction mapping done — {len(y_true_gnn)} valid samples')


In [ ]:
# ─── GNN Predictions on Comparison Sample ────────────────────────────────────
print('Computing GNN predictions...')

gnn_pred_all = []
gnn_prob_all = []

cmp_valid_idx = []
for i, (_, row) in enumerate(cmp_df.iterrows()):
    if row['Y'] not in top5_set:
        continue
    cmp_valid_idx.append(i)

BATCH_SIZE = 32
valid_rows = cmp_df.iloc[cmp_valid_idx].reset_index(drop=True)

for start in range(0, len(valid_rows), BATCH_SIZE):
    batch_rows = valid_rows.iloc[start:start+BATCH_SIZE]
    g1_list, g2_list = [], []
    for _, row in batch_rows.iterrows():
        g1 = smiles_to_gnn_graph(row['Drug1'])
        g2 = smiles_to_gnn_graph(row['Drug2'])
        if g1 is None or g2 is None:
            # Fallback: empty graph
            g1 = Data(x=torch.zeros(1, GNN_NODE_FEAT_DIM),
                      edge_index=torch.zeros(2, 0, dtype=torch.long),
                      fp=torch.zeros(GNN_FP_DIM))
            g2 = g1
        g1_list.append(g1); g2_list.append(g2)
    with torch.no_grad():
        b1 = Batch.from_data_list(g1_list).to(DEVICE)
        b2 = Batch.from_data_list(g2_list).to(DEVICE)
        logits = ddi_gnn(b1, b2)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
        preds  = probs.argmax(axis=1)
    gnn_pred_all.extend(preds)
    gnn_prob_all.extend(probs)

gnn_pred_gnn = np.array(gnn_pred_all)
gnn_prob_arr = np.array(gnn_prob_all)
rf_prob_top5 = np.zeros((len(valid_rows), GNN_N_CLASSES))

# Also build RF probability array over top-5 classes for calibration comparison
for i, (_, row) in enumerate(valid_rows.iterrows()):
    fp1 = smiles_to_fp_rf(row['Drug1'])
    fp2 = smiles_to_fp_rf(row['Drug2'])
    X_r = np.hstack([fp1, fp2]).reshape(1, -1)
    probs_rf_full = ddi_model.predict_proba(X_r)[0]
    for j, cls in enumerate(gnn_le.classes_):
        if cls in set(ddi_le.classes_):
            rf_idx = np.where(ddi_le.classes_ == cls)[0][0]
            rf_prob_top5[i, j] = probs_rf_full[rf_idx]
    # Renormalize
    s = rf_prob_top5[i].sum()
    if s > 0:
        rf_prob_top5[i] /= s

print(f'GNN predictions done — {len(gnn_pred_gnn)} samples')


In [ ]:
# ─── Head-to-Head Metrics ────────────────────────────────────────────────────
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

rf_acc   = accuracy_score(y_true_gnn, rf_pred_gnn)
gnn_acc  = accuracy_score(y_true_gnn, gnn_pred_gnn)
rf_mac   = f1_score(y_true_gnn, rf_pred_gnn,  average='macro',    zero_division=0)
gnn_mac  = f1_score(y_true_gnn, gnn_pred_gnn, average='macro',    zero_division=0)
rf_wt    = f1_score(y_true_gnn, rf_pred_gnn,  average='weighted', zero_division=0)
gnn_wt   = f1_score(y_true_gnn, gnn_pred_gnn, average='weighted', zero_division=0)

metrics_df = pd.DataFrame({
    'Metric':       ['Accuracy', 'Macro F1', 'Weighted F1'],
    'Random Forest': [rf_acc,   rf_mac,      rf_wt],
    'GNN':          [gnn_acc,   gnn_mac,     gnn_wt],
})

print('═' * 55)
print(f'{"Metric":20s}  {"Random Forest":>14s}  {"GNN":>10s}')
print('─' * 55)
for _, row in metrics_df.iterrows():
    rf_v  = f'{row["Random Forest"]:.4f}'
    gnn_v = f'{row["GNN"]:.4f}'
    winner = '← RF' if row['Random Forest'] > row['GNN'] else ('← GNN' if row['GNN'] > row['Random Forest'] else '  tie')
    print(f'{row["Metric"]:20s}  {rf_v:>14s}  {gnn_v:>10s}  {winner}')
print('═' * 55)


In [ ]:
# ─── Per-class F1 Comparison ─────────────────────────────────────────────────
rf_per_cls  = f1_score(y_true_gnn, rf_pred_gnn,  average=None, zero_division=0)
gnn_per_cls = f1_score(y_true_gnn, gnn_pred_gnn, average=None, zero_division=0)

cls_names_short = [GNN_LABEL_MAP.get(i, str(i))[:55] for i in range(GNN_N_CLASSES)]

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
x = np.arange(GNN_N_CLASSES)
w = 0.35

axes[0].barh(x - w/2, rf_per_cls,  w, label='RF',  color='steelblue', alpha=0.85)
axes[0].barh(x + w/2, gnn_per_cls, w, label='GNN', color='coral',     alpha=0.85)
axes[0].set_yticks(x)
axes[0].set_yticklabels(cls_names_short, fontsize=8)
axes[0].set_xlabel('F1 Score'); axes[0].set_xlim(0, 1)
axes[0].set_title('Per-class F1: RF vs GNN')
axes[0].legend()
axes[0].grid(axis='x', alpha=0.3)

# Delta chart: GNN - RF
delta = gnn_per_cls - rf_per_cls
colors_d = ['#4CAF50' if d >= 0 else '#f44336' for d in delta]
axes[1].barh(x, delta, color=colors_d, alpha=0.85)
axes[1].axvline(0, color='white', linewidth=0.8)
axes[1].set_yticks(x)
axes[1].set_yticklabels(cls_names_short, fontsize=8)
axes[1].set_xlabel('GNN F1 − RF F1 (green = GNN better)')
axes[1].set_title('F1 Δ (GNN − RF) per class')
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('RF vs GNN — Per-class F1 on Shared Comparison Set', fontweight='bold')
plt.tight_layout()
plt.savefig('/content/comparison_per_class_f1.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ─── Confusion Matrices Side by Side ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, preds, title in [
    (axes[0], rf_pred_gnn,  'Random Forest'),
    (axes[1], gnn_pred_gnn, 'GNN'),
]:
    cm = confusion_matrix(y_true_gnn, preds, labels=list(range(GNN_N_CLASSES)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=[f'C{i}' for i in range(GNN_N_CLASSES)],
                yticklabels=[f'C{i}' for i in range(GNN_N_CLASSES)],
                ax=ax, linewidths=0.3)
    ax.set_title(f'Confusion Matrix — {title}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

# Legend for class indices
legend_text = '  |  '.join([f'C{i}: {GNN_LABEL_MAP.get(i,str(i))[:35]}' for i in range(GNN_N_CLASSES)])
fig.text(0.5, -0.02, legend_text, ha='center', fontsize=7, color='#aaa', wrap=True)

plt.suptitle('Confusion Matrices — RF vs GNN (top-5 DDI classes)', fontweight='bold')
plt.tight_layout()
plt.savefig('/content/comparison_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ─── Confidence Calibration Comparison ───────────────────────────────────────
# For each model: compare predicted probability for the true class vs actual fraction correct
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, probs, preds, model_name, color in [
    (axes[0], rf_prob_top5, rf_pred_gnn,  'Random Forest', 'steelblue'),
    (axes[1], gnn_prob_arr, gnn_pred_gnn, 'GNN',           'coral'),
]:
    # Max predicted probability per sample (confidence)
    confidence = probs.max(axis=1)
    correct    = (preds == y_true_gnn).astype(int)

    # Bin by confidence
    bins = np.linspace(0, 1, 11)
    bin_idx  = np.digitize(confidence, bins) - 1
    bin_acc  = [correct[bin_idx == b].mean() if (bin_idx == b).sum() > 0 else np.nan
                for b in range(len(bins)-1)]
    bin_mid  = (bins[:-1] + bins[1:]) / 2

    ax.plot([0, 1], [0, 1], 'w--', label='Perfect calibration', alpha=0.5)
    ax.plot(bin_mid, bin_acc, 'o-', color=color, label=model_name)
    ax.set_xlabel('Mean Predicted Confidence')
    ax.set_ylabel('Fraction Correct')
    ax.set_title(f'{model_name} — Calibration')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Confidence Calibration: RF vs GNN', fontweight='bold')
plt.tight_layout()
plt.savefig('/content/comparison_calibration.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ─── Summary Verdict ──────────────────────────────────────────────────────────

def verdict_html(rf_acc, gnn_acc, rf_mac, gnn_mac, rf_wt, gnn_wt,
                 rf_per_cls, gnn_per_cls, cls_names):
    rf_wins  = sum([rf_acc > gnn_acc, rf_mac > gnn_mac, rf_wt > gnn_wt])
    gnn_wins = sum([gnn_acc > rf_acc, gnn_mac > rf_mac, gnn_wt > rf_wt])
    rf_per_wins  = (rf_per_cls > gnn_per_cls).sum()
    gnn_per_wins = (gnn_per_cls > rf_per_cls).sum()

    if gnn_wins > rf_wins:
        overall = '🧠 <b>GNN</b> wins on aggregate metrics'
        color   = '#1a2a3a'
    elif rf_wins > gnn_wins:
        overall = '🌲 <b>Random Forest</b> wins on aggregate metrics'
        color   = '#1b3a1b'
    else:
        overall = '🤝 <b>Tied</b> on aggregate metrics'
        color   = '#2a2a2a'

    best_rf_class  = cls_names[np.argmax(rf_per_cls  - gnn_per_cls)]
    best_gnn_class = cls_names[np.argmax(gnn_per_cls - rf_per_cls)]

    return f"""
    <div style="background:{color};border-radius:10px;padding:18px 24px;color:#eee;margin:10px 0">
      <h3 style="margin:0 0 10px 0">🏆 Comparison Verdict</h3>
      <p><b>Overall:</b> {overall}</p>
      <ul>
        <li>RF accuracy: <b>{rf_acc:.4f}</b> vs GNN accuracy: <b>{gnn_acc:.4f}</b></li>
        <li>RF macro F1: <b>{rf_mac:.4f}</b> vs GNN macro F1: <b>{gnn_mac:.4f}</b></li>
        <li>RF wins {rf_per_wins}/{len(cls_names)} per-class F1 battles; GNN wins {gnn_per_wins}/{len(cls_names)}</li>
        <li>RF best class advantage: <em>{best_rf_class[:60]}</em></li>
        <li>GNN best class advantage: <em>{best_gnn_class[:60]}</em></li>
      </ul>
      <p style="margin-top:10px;font-size:0.9em;color:#bbb">
        <b>Interpretation:</b> The Random Forest uses 4096-dimensional Morgan fingerprint
        pair features trained on all 89 classes — it benefits from a larger training
        set and explicit structural bits. The GNN captures subgraph patterns end-to-end
        with only the top-5 classes; it generalises better on structurally diverse drugs
        but may underfit rare interaction types. Consider using RF for quick broad
        screening (all 89 classes) and the GNN for deeper analysis of the most
        common interaction mechanisms.
      </p>
    </div>
    """

from IPython.display import HTML
display(HTML(verdict_html(
    rf_acc, gnn_acc, rf_mac, gnn_mac, rf_wt, gnn_wt,
    rf_per_cls, gnn_per_cls, cls_names_short
)))


---
## Programmatic API


In [ ]:
# ─── Programmatic API Example ────────────────────────────────────────────────
ASPIRIN  = 'CC(=O)Oc1ccccc1C(=O)O'
WARFARIN = 'CC(=O)Oc1ccc(cc1)n1ccnc1'

print('=' * 60)
print('MODULE A: DDI Random Forest — Interaction Mechanism (top 3)')
print('=' * 60)
for desc, prob in predict_ddi_type(ASPIRIN, WARFARIN, top_k=3):
    print(f'  {prob:.3f}  {desc[:90]}')

print()
print('=' * 60)
print('MODULE B: DDI GNN — Interaction Prediction (top 3)')
print('=' * 60)
for desc, prob in predict_ddi_gnn(ASPIRIN, WARFARIN, top_k=3):
    print(f'  {prob:.3f}  {desc[:90]}')

print()
print('=' * 60)
print('MODULE C: Food Behavior (Lookup Table)')
print('=' * 60)
for name in ['Aspirin', 'Warfarin']:
    info, matched = lookup_food_by_name(name)
    if info:
        print(f'  {matched}:')
        for line in info[:3]:
            print(f'    - {line}')

print()
print('=' * 60)
print('Name → SMILES via PubChem')
print('=' * 60)
for drug in ['Ibuprofen', 'Metformin', 'Atorvastatin']:
    smi = name_to_smiles(drug)
    print(f'  {drug}: {smi}')
